# 📊 US Superstore Data Visualization
## Interactive Data Analysis with Matplotlib and Seaborn

**Dataset:** US Superstore Sales (2014–2017)  
**Tools:** Pandas · Matplotlib · Seaborn  
**Author:** Data Analysis Bootcamp – Daily Challenge W5D5


## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

# ── Global style ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})
sns.set_theme(style="whitegrid", palette="muted")

print("Libraries loaded ✓")


## 2. Data Loading & Cleaning

The dataset contains **9 994 transactions** across four years (2014–2017).  
All columns are already well-typed after conversion from `.xls`; the only
preprocessing needed is:

- Parse `Order Date` / `Ship Date` as datetime (already done by `read_excel`)
- Extract `Year` and `Month` features for time-series analysis
- Fix `Discount` – stored as an integer percentage (0–80), so divide by 100


In [ ]:
FILE = "US_Superstore_data.xlsx"   # place in the same folder as this notebook

df = pd.read_excel(FILE, engine="openpyxl")

# Feature engineering
df["Year"]  = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.to_period("M")

# Discount was stored as 0-80 integer; normalise to 0.0-0.8
if df["Discount"].max() > 1:
    df["Discount"] = df["Discount"] / 100

# Quick sanity check
print(f"Rows: {len(df):,}  |  Columns: {df.shape[1]}")
print(f"Date range : {df['Order Date'].min().date()} → {df['Order Date'].max().date()}")
print(f"Null values: {df.isnull().sum().sum()}")
df[["Sales","Quantity","Discount","Profit"]].describe().round(2)


## 3. Matplotlib Visualizations

### 3a. Interactive Annual Sales Trend

We aggregate sales by year and plot an annotated line chart with markers.


In [ ]:
annual = df.groupby("Year")["Sales"].sum().reset_index()

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(annual["Year"], annual["Sales"],
        marker="o", linewidth=2.5, markersize=9,
        color="#2563EB", markerfacecolor="white", markeredgewidth=2.5)

# Annotate each data point
for _, row in annual.iterrows():
    ax.annotate(f"${row['Sales']/1e6:.2f}M",
                xy=(row["Year"], row["Sales"]),
                xytext=(0, 14), textcoords="offset points",
                ha="center", fontsize=10, fontweight="bold", color="#1E40AF")

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e6:.1f}M"))
ax.set_xticks(annual["Year"])
ax.set_title("Annual Sales Trend  (2014 – 2017)")
ax.set_xlabel("Year")
ax.set_ylabel("Total Sales")
ax.fill_between(annual["Year"], annual["Sales"], alpha=0.08, color="#2563EB")

plt.tight_layout()
plt.savefig("fig1_sales_trend.png", bbox_inches="tight")
plt.show()
print("Figure saved → fig1_sales_trend.png")


### 3b. Sales Distribution by US State (Choropleth-style Bar Map)

Because all orders are from the United States we build a ranked bar chart of
the **top 15 states by sales** – a simpler and clearer alternative to a full
choropleth when the dataset has only one country.


In [ ]:
state_sales = (df.groupby("State")["Sales"]
                 .sum()
                 .sort_values(ascending=False)
                 .head(15))

fig, ax = plt.subplots(figsize=(11, 6))

colors = ["#1D4ED8" if i == 0 else "#3B82F6" if i < 3 else "#93C5FD"
          for i in range(len(state_sales))]

bars = ax.barh(state_sales.index[::-1], state_sales.values[::-1], color=colors[::-1], edgecolor="white")

for bar, val in zip(bars, state_sales.values[::-1]):
    ax.text(bar.get_width() + 2000, bar.get_y() + bar.get_height()/2,
            f"${val:,.0f}", va="center", fontsize=9)

ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}K"))
ax.set_title("Top 15 US States by Total Sales")
ax.set_xlabel("Total Sales")
ax.set_ylabel("")

plt.tight_layout()
plt.savefig("fig2_state_sales.png", bbox_inches="tight")
plt.show()
print("Figure saved → fig2_state_sales.png")


## 4. Seaborn Visualizations

### 4a. Top 10 Products by Sales


In [ ]:
top10 = (df.groupby("Product Name")["Sales"]
           .sum()
           .sort_values(ascending=False)
           .head(10)
           .reset_index())
top10["Short Name"] = top10["Product Name"].str[:40] + "…"

fig, ax = plt.subplots(figsize=(11, 6))
palette = sns.color_palette("Blues_r", n_colors=10)

sns.barplot(data=top10, x="Sales", y="Short Name",
            hue="Short Name", palette=palette, legend=False, ax=ax)

for bar, val in zip(ax.patches, top10["Sales"]):
    ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
            f"${val:,.0f}", va="center", fontsize=9)

ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}K"))
ax.set_title("Top 10 Products by Total Sales")
ax.set_xlabel("Total Sales")
ax.set_ylabel("")

plt.tight_layout()
plt.savefig("fig3_top10_products.png", bbox_inches="tight")
plt.show()
print("Figure saved → fig3_top10_products.png")


### 4b. Profit vs Discount – Scatter Plot by Category


In [ ]:
sample = df.sample(n=min(3000, len(df)), random_state=42)

fig, ax = plt.subplots(figsize=(10, 6))

palette = {"Furniture":"#F59E0B", "Office Supplies":"#10B981", "Technology":"#3B82F6"}

for cat, grp in sample.groupby("Category"):
    ax.scatter(grp["Discount"], grp["Profit"],
               alpha=0.45, s=25, color=palette[cat], label=cat)

# Regression line across all data
m, b = np.polyfit(sample["Discount"], sample["Profit"], 1)
x_line = np.linspace(sample["Discount"].min(), sample["Discount"].max(), 200)
ax.plot(x_line, m*x_line + b, color="#EF4444", linewidth=2,
        linestyle="--", label=f"Trend  (slope={m:+.0f})")

ax.axhline(0, color="grey", linewidth=0.8, linestyle=":")
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_title("Profit vs Discount by Category")
ax.set_xlabel("Discount Rate")
ax.set_ylabel("Profit ($)")
ax.legend(title="Category", framealpha=0.9)

plt.tight_layout()
plt.savefig("fig4_profit_discount.png", bbox_inches="tight")
plt.show()
print("Figure saved → fig4_profit_discount.png")


## 5. Comparative Analysis – Matplotlib vs Seaborn

| Dimension | Matplotlib | Seaborn |
|-----------|-----------|---------|
| **API style** | Imperative – you control every element | Declarative – high-level wrappers |
| **Default aesthetics** | Minimal; good for publication plots | Theme-aware; polished out of the box |
| **Statistical features** | None built-in | Regression lines, violin plots, pair-grids |
| **Customisation ceiling** | Unlimited; access to every artist | Excellent, but sometimes fights you on fine-grained edits |
| **Best use case** | Engineering dashboards, custom layouts | Exploratory data analysis, statistical storytelling |

### Key Insights from the Visualizations

1. **Sales grow every year** – revenue climbed from **$484K (2014)** to **$733K (2017)**, a ~51% increase over three years.
2. **California dominates** – CA alone accounts for more total sales than the next two states combined.
3. **Technology rules the top products** – the highest-revenue SKUs are almost entirely phones and copiers.
4. **Discounts destroy profit** – the scatter plot shows a clear negative trend: as discount rate increases, profit reliably drops, often turning negative above 30–40% discount.
5. **Furniture is the most discount-damaged category** – orange points cluster in the loss zone at higher discounts.

### Recommendation
Apply discounts selectively and cap them at **20%** to preserve profitability.
